# Actividad 3A. Cuaderno Jupyter: Segmentación de imágenes con OpenCV
**Procesamiento Digital de Imágenes · MCDI · INFOTEC · 2027-1** · Unidad 3

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egonzalez1989/pdi-mcdi/blob/main/plantillas/3A_Plantilla_Segmentacion.ipynb)

| | |
|---|---|
| **Estudiante** | *Escribe aquí tu nombre completo* |
| **Tipo de actividad** | Individual |
| **Fecha límite** | Domingo 7 de marzo de 2027, 23:59 hrs. |
| **Puntaje** | 20 puntos |

**Propósito.** Implementar y comparar métodos de segmentación por bordes, umbralización y agrupamiento, evaluándolos visual y cuantitativamente.

**Producto esperado.** Una libreta Jupyter bien documentada que incluya:
- Detección de bordes con al menos dos métodos (Sobel y Canny).
- Umbralización global y de Otsu o adaptativa para separar objetos del fondo.
- Segmentación por agrupamiento de intensidades (k-means).
- Comparación visual y cuantitativa (IoU y Dice) de los métodos, con análisis en Markdown.

### Instrucciones
1. Descarga el cuaderno plantilla 3A_Plantilla_Segmentacion.ipynb. Selecciona una o dos imágenes con objetos diferenciables del fondo (piezas, semillas, monedas, objetos sobre una mesa); puedes usar las de la carpeta datos/segmentacion del repositorio, que incluyen máscara de referencia.
2. Segmentación por discontinuidades: aplica Sobel y visualiza la magnitud del gradiente; aplica Canny con distintos pares de umbrales y comenta qué parámetros producen bordes más útiles.
3. Segmentación por similaridad: aplica umbralización global con al menos dos valores, el método de Otsu y, si es posible, umbralización adaptativa; compara las máscaras.
4. Agrupamiento: reordena los píxeles como vector de intensidades, aplica k-means con K = 2 o K = 3 y reconstruye la imagen segmentada a partir de las etiquetas.
5. Evaluación cuantitativa: compara las máscaras obtenidas contra la máscara de referencia (la del repositorio o una que dibujes en GIMP) con IoU y Dice, y presenta una tabla comparativa.
6. Análisis comparativo: coloca en una misma figura las salidas de bordes, umbralización y k-means y comenta qué método separa mejor los objetos, qué errores aparecen (bordes falsos, regiones fragmentadas, pérdida de detalle) y qué ajustes harías en una siguiente iteración.
7. Al final de la libreta, completa la declaración de uso de IA (herramienta, para qué la usaste y qué partes son resultado directo de ella) o indica que no la utilizaste. Tu entrega puede ser seleccionada para una defensa oral.
8. Guarda tu libreta como 3A_Nombre_Apellido.ipynb, expórtala a PDF con el mismo nombre y entrega ambos archivos.

### Lista de cotejo (así se evaluará tu trabajo)
| Criterio | Puntaje |
|---|---|
| Implementación de segmentación por bordes (Sobel/Canny). | 5 |
| Implementación de umbralización (global y Otsu/adaptativa). | 5 |
| Implementación de segmentación basada en intensidades (p. ej., k-means). | 4 |
| Claridad en código, visualizaciones y análisis comparativo en Markdown, con evaluación IoU/Dice contra la referencia. | 6 |
| **Total** | **20** |

> 💡 Escribe tus interpretaciones en celdas Markdown como esta. Una figura sin interpretación no demuestra aprendizaje. Si usas código de otras fuentes, cítalo en Referencias; si usas IA, decláralo en la sección final.

In [ ]:
# Configuración del entorno. En Google Colab todas estas bibliotecas ya vienen instaladas.
import sys, platform
import numpy as np
import matplotlib.pyplot as plt
import cv2
import skimage
from skimage import data

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "image.cmap": "gray"})
np.random.seed(42)

def mostrar(imagenes, titulos=None, cols=None, cmap="gray", tam=3.2, ejes=False):
    """Muestra una lista de imágenes en una cuadrícula. Las imágenes a color deben estar en RGB."""
    if not isinstance(imagenes, (list, tuple)):
        imagenes = [imagenes]
    n = len(imagenes)
    cols = cols or min(n, 4)
    filas = int(np.ceil(n / cols))
    fig, axs = plt.subplots(filas, cols, figsize=(tam * cols, tam * filas), squeeze=False)
    for i, ax in enumerate(axs.flat):
        if i < n:
            im = imagenes[i]
            ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
            if titulos:
                ax.set_title(titulos[i])
        if not ejes or i >= n:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Python {platform.python_version()} | NumPy {np.__version__} | OpenCV {cv2.__version__} | scikit-image {skimage.__version__}")

# Datos del curso: se buscan en la carpeta datos/ del repositorio (si lo clonaste) o se descargan.
import os, io, zipfile, urllib.request
def carpeta_datos(url="https://github.com/egonzalez1989/pdi-mcdi/archive/refs/heads/main.zip"):
    for d in ("datos", "../datos", "pdi-mcdi-main/datos"):
        if os.path.isdir(d):
            return d
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        return "pdi-mcdi-main/datos"
    except Exception as e:
        print("No se pudieron descargar los datos del curso:", e)
        return None
DATOS = carpeta_datos()
print("Carpeta de datos:", DATOS)

import pandas as pd
from sklearn.cluster import KMeans

## 0. Imagen y máscara de referencia
Por omisión se usan `datos/segmentacion/monedas.png` y su máscara. También puedes usar `objetos_sinteticos.png` o tu propia imagen con una máscara dibujada en GIMP (blanco = objeto).

In [ ]:
RUTA_IMAGEN, RUTA_MASCARA = "", ""
if not RUTA_IMAGEN and DATOS:
    RUTA_IMAGEN = os.path.join(DATOS, "segmentacion", "monedas.png")
    RUTA_MASCARA = os.path.join(DATOS, "segmentacion", "monedas_mascara.png")
gris = cv2.imread(RUTA_IMAGEN, cv2.IMREAD_GRAYSCALE) if RUTA_IMAGEN else None
gt = cv2.imread(RUTA_MASCARA, cv2.IMREAD_GRAYSCALE) > 127 if RUTA_MASCARA else None
if gris is None:   # respaldo sin conexión (sin referencia)
    gris, gt = data.coins(), None
mostrar([gris] + ([gt] if gt is not None else []), ["Imagen", "Referencia"][: 1 + (gt is not None)])

def iou(p, g):
    p, g = p.astype(bool), g.astype(bool); return (p & g).sum() / max((p | g).sum(), 1)
def dice(p, g):
    p, g = p.astype(bool), g.astype(bool); return 2 * (p & g).sum() / max(p.sum() + g.sum(), 1)

## 1. Segmentación por discontinuidades: Sobel y Canny

In [ ]:
# TODO: magnitud del gradiente con cv2.Sobel (dx y dy) y Canny con al menos tres pares de umbrales

*¿Qué parámetros producen bordes más útiles para tu caso?* ...

## 2. Segmentación por similaridad: umbralización

In [ ]:
mascaras = {}   # guarda aquí cada máscara binaria (objeto = True) para evaluarla en la sección 4
# TODO: umbral global con dos valores, Otsu y adaptativo (cv2.adaptiveThreshold); p. ej.:
# _, m = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU); mascaras["Otsu"] = m > 0

## 3. Agrupamiento de intensidades: k-means

In [ ]:
# TODO: X = gris.reshape(-1, 1); KMeans(n_clusters=2 o 3, n_init=10, random_state=0).fit(X)
#       reconstruye la imagen de etiquetas y conviértela en máscara (el grupo más claro = objeto)

## 4. Evaluación cuantitativa contra la referencia

In [ ]:
if gt is not None and mascaras:
    tabla = pd.DataFrame([(k, iou(m, gt), dice(m, gt)) for k, m in mascaras.items()], columns=["Método", "IoU", "Dice"])
    display(tabla.sort_values("IoU", ascending=False).round(3))
    mostrar(list(mascaras.values()), list(mascaras.keys()))

## 5. Análisis comparativo
- ¿Qué método separa mejor los objetos del fondo?
- ¿Qué errores aparecen (bordes falsos, regiones fragmentadas, pérdida de detalle)?
- ¿Qué ajustes harías en una siguiente iteración?

## Referencias
*Lista en formato APA las fuentes consultadas (libros, artículos, documentación y código de terceros).*

## Declaración de uso de inteligencia artificial
*Obligatoria. Completa la tabla o escribe: «No utilicé herramientas de IA en esta actividad».*

| Herramienta (nombre y versión) | Para qué la usé | Partes del trabajo que son resultado directo de la IA |
|---|---|---|
| ... | ... | ... |

> **Regla de oro:** entrega solo código y análisis que puedas explicar y modificar línea por línea. Tu entrega puede ser seleccionada (de forma pseudoaleatoria) para una **defensa oral** por videollamada, sin asistentes de IA, con 4 a 5 preguntas sobre tu código, la lógica de tu solución y la teoría. Consulta el documento «Uso adecuado de la IA».

---
**Antes de entregar:** `Entorno de ejecución → Reiniciar y ejecutar todo`, verifica que no haya errores y exporta a PDF (`Archivo → Imprimir → Guardar como PDF` o `jupyter nbconvert --to webpdf`).